# Fixed universal attacks evaluated on AA-CLIP

This notebook mirrors the complete AnomalyCLIP Kaggle evaluation workflow for the official [AA-CLIP](https://github.com/Mwxinnn/AA-CLIP) implementation. It clones pinned source, resolves the exact canonical perturbation archive and fixed target IDs, uses paper-default AA-CLIP inference, loads the supplied [AA-CLIP checkpoint dataset](https://www.kaggle.com/datasets/parsagh1383/aa-clip-checkpoints-main), reports clean/adversarial metrics, exports predictions and qualitative samples, and packages both outputs. Enable a GPU and Internet before running all cells.

The zero-shot mapping is deliberate: MVTec is evaluated with `TrainOnVisA`, while VisA is evaluated with `TrainOnMVTec`. Attach the frozen q95 outputs produced by `kaggle_new_aaclip_thresholds.ipynb` as an additional Kaggle input.

In [ ]:
import hashlib
import shutil
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
AACLIP_ROOT = WORKING / 'AA-CLIP'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
AACLIP_REPO_URL = 'https://github.com/Mwxinnn/AA-CLIP.git'
AACLIP_COMMIT = '53db195f230442aa118c246876c94ba1c76139cc'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(AACLIP_REPO_URL, AACLIP_ROOT, AACLIP_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'einops==0.7.0', 'ftfy==6.2.0', 'ipdb>=0.13', 'kornia==0.6.9',
    'tiktoken==0.7.0', 'timm==0.6.12'
], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

# AA-CLIP's factory expects this exact OpenAI ViT-L/14@336px file in model/.
BASE_MODEL_NAME = 'ViT-L-14-336px.pt'
BASE_MODEL_SHA256 = '3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02'
BASE_MODEL_URL = (
    'https://openaipublic.azureedge.net/clip/models/'
    + BASE_MODEL_SHA256 + '/' + BASE_MODEL_NAME
)
BASE_MODEL_PATH = AACLIP_ROOT / 'model' / BASE_MODEL_NAME

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

attached_base = next(
    (path for path in Path('/kaggle/input').rglob(BASE_MODEL_NAME) if path.is_file()),
    None,
)
if not BASE_MODEL_PATH.is_file() or sha256(BASE_MODEL_PATH) != BASE_MODEL_SHA256:
    if attached_base is not None:
        if sha256(attached_base) != BASE_MODEL_SHA256:
            raise RuntimeError(f'Attached base model has the wrong SHA256: {attached_base}')
        shutil.copy2(attached_base, BASE_MODEL_PATH)
    else:
        import torch
        torch.hub.download_url_to_file(
            BASE_MODEL_URL, str(BASE_MODEL_PATH), hash_prefix=BASE_MODEL_SHA256, progress=True
        )
if sha256(BASE_MODEL_PATH) != BASE_MODEL_SHA256:
    raise RuntimeError(f'Base-model checksum mismatch: {BASE_MODEL_PATH}')
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')
print('Official AA-CLIP:', AACLIP_ROOT)
print('Verified base model:', BASE_MODEL_PATH)

In [ ]:
import gdown

from blackbox_evaluation_pipeline.universal_eval.artifacts import load_manifest

print('===== STEP 2: RESOLVE THE EXACT CANONICAL ARTIFACT ARCHIVE =====')
DRIVE_FILE_ID = '10ZiaDs6u5G_WFVbrd9tt-Ug6Dy_aaFS5'
ARTIFACT_DIR_NAME = 'canonical_clip_universal_attacks_full'

def valid_artifact_root(path):
    return path.is_dir() and (path / 'all_canonical_attack_artifacts.json').is_file()

candidates = [
    EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'perturbations' / ARTIFACT_DIR_NAME,
    WORKING / ARTIFACT_DIR_NAME,
]
if Path('/kaggle/input').is_dir():
    candidates.extend(Path('/kaggle/input').rglob(ARTIFACT_DIR_NAME))
ARTIFACTS_ROOT = next((path for path in candidates if valid_artifact_root(path)), None)

if ARTIFACTS_ROOT is None:
    archive_path = WORKING / 'canonical_clip_universal_attacks_full_ARTIFACTS.zip'
    print('Artifacts are not attached as a Kaggle input; downloading the exact Drive file...')
    downloaded = gdown.download(id=DRIVE_FILE_ID, output=str(archive_path), quiet=False)
    if not downloaded or not archive_path.is_file():
        raise RuntimeError('Drive download failed. Attach the ZIP as a private Kaggle dataset and rerun.')
    extract_root = WORKING / 'canonical_artifact_extract'
    extract_root.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archive_path), str(extract_root))
    extracted = [path for path in extract_root.rglob(ARTIFACT_DIR_NAME) if valid_artifact_root(path)]
    if not extracted and valid_artifact_root(extract_root):
        extracted = [extract_root]
    if len(extracted) != 1:
        raise RuntimeError(f'Expected one extracted artifact root, found: {extracted}')
    ARTIFACTS_ROOT = extracted[0]

artifacts = load_manifest(ARTIFACTS_ROOT)
print('Canonical root:', ARTIFACTS_ROOT)
print('Available conditions:', len(artifacts))
for source, target in sorted({(a.record['source_dataset'], a.record['target_dataset']) for a in artifacts}):
    count = sum(a.record['source_dataset'] == source and a.record['target_dataset'] == target for a in artifacts)
    print(f'  {source} -> {target}: {count}')

In [ ]:
import torch

print('===== STEP 3: RESOLVE DATASETS AND ZERO-SHOT AA-CLIP CHECKPOINTS =====')
CHECKPOINT_DATASET_URL = 'https://www.kaggle.com/datasets/parsagh1383/aa-clip-checkpoints-main'

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

checkpoint_roots = [
    path for path in (
        Path('/kaggle/input/aa-clip-checkpoints-main'),
        Path('/kaggle/input/datasets/parsagh1383/aa-clip-checkpoints-main'),
    ) if path.is_dir()
]
if not checkpoint_roots:
    raise FileNotFoundError(
        'Attach parsagh1383/aa-clip-checkpoints-main as a Kaggle input: ' + CHECKPOINT_DATASET_URL
    )

def resolve_training_checkpoint(training_name):
    matches = []
    for root in checkpoint_roots:
        for image_path in root.rglob('image_adapter.pth'):
            if training_name.lower() in {part.lower() for part in image_path.parts}:
                matches.append(image_path.parent)
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {training_name} checkpoint directory, found: {matches}')
    directory = matches[0]
    image_path = directory / 'image_adapter.pth'
    text_path = directory / 'text_adapter.pth'
    return image_path, text_path if text_path.is_file() else None

TRAIN_ON_MVTEC_IMAGE, TRAIN_ON_MVTEC_TEXT = resolve_training_checkpoint('TrainOnMVTec')
TRAIN_ON_VISA_IMAGE, TRAIN_ON_VISA_TEXT = resolve_training_checkpoint('TrainOnVisA')

# Opposite-dataset weights preserve AA-CLIP's zero-shot protocol.
MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(AACLIP_ROOT),
        'image_checkpoint_path': str(TRAIN_ON_VISA_IMAGE),
        'text_checkpoint_path': str(TRAIN_ON_VISA_TEXT) if TRAIN_ON_VISA_TEXT else None,
        'target_dataset': 'mvtec',
    },
    'visa': {
        'repository_root': str(AACLIP_ROOT),
        'image_checkpoint_path': str(TRAIN_ON_MVTEC_IMAGE),
        'text_checkpoint_path': str(TRAIN_ON_MVTEC_TEXT) if TRAIN_ON_MVTEC_TEXT else None,
        'target_dataset': 'visa',
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('MVTec target <- TrainOnVisA:', TRAIN_ON_VISA_IMAGE, TRAIN_ON_VISA_TEXT)
print('VisA target <- TrainOnMVTec:', TRAIN_ON_MVTEC_IMAGE, TRAIN_ON_MVTEC_TEXT)

In [ ]:
import json

from blackbox_evaluation_pipeline import EvaluationConfig, run_evaluation

print('===== STEP 4: RUN FIXED-ID CLEAN/ADVERSARIAL EVALUATION =====')
FULL_RUN = True
OUTPUT_ROOT = WORKING / ('kaggle_new_aaclip_full' if FULL_RUN else 'kaggle_new_aaclip_check')
SAMPLES_ROOT = WORKING / (
    'kaggle_new_aaclip_samples_full' if FULL_RUN else 'kaggle_new_aaclip_samples_check'
)

def normalized_name(value):
    return ''.join(character for character in value.lower() if character.isalnum())

def find_frozen_threshold(dataset):
    candidates = []
    committed = (
        EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'thresholds'
        / 'aaclip_thresholds_q95' / dataset / 'category_thresholds.json'
    )
    if committed.is_file():
        candidates.append(committed)
    candidates.extend(Path('/kaggle/input').rglob('category_thresholds.json'))
    valid = []
    for path in candidates:
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if (
            payload.get('dataset') == dataset
            and normalized_name(str(payload.get('target_model', ''))) == 'aaclip'
            and payload.get('threshold_mode') == 'normal_train_quantile'
        ):
            valid.append(path)
    if len(valid) != 1:
        raise RuntimeError(
            f'Expected exactly one frozen AA-CLIP threshold file for {dataset}, found {valid}. '
            'Run kaggle_new_aaclip_thresholds.ipynb, publish its output as a Kaggle dataset, '
            'and attach that dataset to this notebook.'
        )
    return str(valid[0])

THRESHOLDS_BY_TARGET = {dataset: find_frozen_threshold(dataset) for dataset in ('mvtec', 'visa')}
print('Frozen thresholds:', THRESHOLDS_BY_TARGET)

config = EvaluationConfig(
    artifacts_root=str(ARTIFACTS_ROOT),
    mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT),
    output_root=str(OUTPUT_ROOT),
    model_name='aaclip',
    model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    thresholds_by_target=THRESHOLDS_BY_TARGET,
    device='cuda',
    batch_size=2,
    metric_size=518,
    # Official AA-CLIP already applies the industrial 7x7, sigma=1 blur per level.
    anomaly_map_sigma=0.0,
    aupro_fpr_limit=0.30,
    aupro_max_thresholds=200,
    verify_checksums=True,
    save_predictions=True,
    save_qualitative_samples=True,
    qualitative_output_root=str(SAMPLES_ROOT),
    max_conditions=None if FULL_RUN else 1,
    run_notes=(
        'Exact canonical Drive artifacts; fixed manifest evaluation IDs; AA-CLIP official '
        'ViT-L-14-336/518 defaults; opposite-dataset zero-shot adapter weights.'
    ),
)
SUMMARY_PATH = run_evaluation(config)
print('Finished:', SUMMARY_PATH)

In [ ]:
import csv

print('===== STEP 5: PREVIEW SUMMARY =====')
with SUMMARY_PATH.open(newline='', encoding='utf-8') as handle:
    summary_rows = list(csv.DictReader(handle))
columns = [
    'source_dataset', 'target_dataset', 'direction', 'loss_mode',
    'clean_i_auroc', 'adversarial_i_auroc', 'delta_i_auroc',
    'clean_p_auroc', 'adversarial_p_auroc', 'delta_p_auroc',
    'clean_aupro', 'adversarial_aupro', 'delta_aupro',
    'clean_accuracy', 'adversarial_accuracy',
    'clean_fpr', 'adversarial_fpr', 'clean_fnr', 'adversarial_fnr',
    'attack_flip_rate', 'targeted_attack_success_rate',
]
for row in summary_rows:
    print({column: row[column] for column in columns})

In [ ]:
print('===== STEP 6: PACKAGE OUTPUTS =====')
results_archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
samples_archive = shutil.make_archive(
    str(SAMPLES_ROOT), 'zip', root_dir=SAMPLES_ROOT.parent, base_dir=SAMPLES_ROOT.name
)
print('Packaged numerical results:', results_archive)
print('Packaged qualitative samples:', samples_archive)